Install and Import Dependencies

In [ ]:
!pip install -q datasets pandas numpy

import os
import ast
import numpy as np
import pandas as pd
from datasets import load_dataset
from collections import defaultdict

print("Dependencies loaded successfully!")

Dependencies loaded successfully!


Download and Save the Dataset

In [ ]:
os.makedirs("data", exist_ok=True)
csv_path = "data/chordonomicon_v2.csv"

if not os.path.exists(csv_path):
    print("Downloading Chordonomicon from Hugging Face...")
    ds = load_dataset("ailsntua/Chordonomicon")
    df = ds['train'].to_pandas()
    df.to_csv(csv_path, index=False)
    print(f"Saved dataset to {csv_path}")
else:
    print(f"Dataset already exists locally at {csv_path}")

Dataset already exists locally at data/chordonomicon_v2.csv


Preprocess Chord Sequences

In [ ]:
print("Preprocessing sequences...")
df = pd.read_csv(csv_path)

sequences = []
for seq in df['chords']:
    if isinstance(seq, str):
        # Split by spaces and filter out section/metadata tags (e.g., <intro_1>)
        parsed_seq = [chord for chord in seq.split() if not (chord.startswith('<') and chord.endswith('>'))]
        if len(parsed_seq) > 1:
            sequences.append(parsed_seq)

print(f"Loaded {len(sequences)} valid chord sequences.")

Preprocessing sequences...


/tmp/ipykernel_2710/783725919.py:2: DtypeWarning: Columns (2,3,5,6,7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


Loaded 679681 valid chord sequences.


Split into Train, Validation, and Test Sets

In [ ]:
np.random.seed(42)
indices = np.random.permutation(len(sequences))

train_idx = int(0.8 * len(sequences))
val_idx = int(0.9 * len(sequences))

train_sequences = [sequences[i] for i in indices[:train_idx]]
val_sequences = [sequences[i] for i in indices[train_idx:val_idx]]
test_sequences = [sequences[i] for i in indices[val_idx:]]

print(f"Split sizes -> Train: {len(train_sequences)}, Val: {len(val_sequences)}, Test: {len(test_sequences)}")

Split sizes -> Train: 543744, Val: 67968, Test: 67969


Train the Markov / N-gram Baseline

In [ ]:
def train_markov_baseline(train_data, n=2):
    """
    Trains an N-gram transition model.
    n=2: Bigram (1 context chord -> 1 target chord)
    n=3: Trigram (2 context chords -> 1 target chord)
    """
    transition_counts = defaultdict(lambda: defaultdict(int))

    for seq in train_data:
        for i in range(len(seq) - (n - 1)):
            context = tuple(seq[i:i + n - 1])
            target = seq[i + n - 1]
            transition_counts[context][target] += 1

    # Normalize counts to probabilities
    transition_probs = {}
    for context, targets in transition_counts.items():
        total_count = sum(targets.values())
        transition_probs[context] = {
            chord: count / total_count for chord, count in targets.items()
        }

    return transition_probs

# Train a Bigram model (n=2)
print("Training Markov baseline model...")
markov_model = train_markov_baseline(train_sequences, n=2)
print(f"Model trained with {len(markov_model)} unique contexts.")

Training Markov baseline model...
Model trained with 4093 unique contexts.


Evaluate the Baseline on the Test Set (Using Random Sample)

In [ ]:
import random

def evaluate_baseline(test_data, model_probs, n=2, top_k=3, sample_size=10000):
    top_1_correct = 0
    top_k_correct = 0
    total_predictions = 0

    # Take a random sample of test sequences (runs faster)
    if len(test_data) > sample_size:
        eval_data = random.sample(test_data, sample_size)
    else:
        eval_data = test_data

    print(f"Evaluating on a sample of {len(eval_data)} test sequences...")

    for seq in eval_data:
        for i in range(len(seq) - (n - 1)):
            context = tuple(seq[i:i + n - 1])
            actual_next_chord = seq[i + n - 1]
            total_predictions += 1

            if context not in model_probs:
                continue

            # Sort predicted next chords by highest probability
            sorted_preds = sorted(
                model_probs[context].items(),
                key=lambda x: x[1],
                reverse=True
            )
            top_preds = [chord for chord, prob in sorted_preds[:top_k]]

            if len(top_preds) > 0 and top_preds[0] == actual_next_chord:
                top_1_correct += 1
            if actual_next_chord in top_preds:
                top_k_correct += 1

    top_1_acc = top_1_correct / total_predictions if total_predictions > 0 else 0
    top_k_acc = top_k_correct / total_predictions if total_predictions > 0 else 0
    return top_1_acc, top_k_acc

top_1, top_3 = evaluate_baseline(test_sequences, markov_model, n=2, top_k=3, sample_size=10000)

print("\n Evaluation Results:")
print(f"Baseline Top-1 Accuracy: {top_1:.4f}")
print(f"Baseline Top-3 Accuracy: {top_3:.4f}")

Evaluating on a sample of 10000 test sequences...

--- Evaluation Results (Sampled) ---
Baseline Top-1 Accuracy: 0.2471
Baseline Top-3 Accuracy: 0.5318


Evaluate the Baseline on the Test Set (Using Full Set)

In [ ]:
def evaluate_baseline(test_data, model_probs, n=2, top_k=3):
    top_1_correct = 0
    top_k_correct = 0
    total_predictions = 0

    for seq in test_data:
        for i in range(len(seq) - (n - 1)):
            context = tuple(seq[i:i + n - 1])
            actual_next_chord = seq[i + n - 1]
            total_predictions += 1

            if context not in model_probs:
                continue


            # Sort predicted next chords by highest probability
            sorted_preds = sorted(
                model_probs[context].items(),
                key=lambda x: x[1],
                reverse=True
            )
            top_preds = [chord for chord, prob in sorted_preds[:top_k]]

            if len(top_preds) > 0 and top_preds[0] == actual_next_chord:
                top_1_correct += 1
            if actual_next_chord in top_preds:
                top_k_correct += 1

    top_1_acc = top_1_correct / total_predictions if total_predictions > 0 else 0
    top_k_acc = top_k_correct / total_predictions if total_predictions > 0 else 0
    return top_1_acc, top_k_acc

print("Evaluating baseline on test set...")
top_1, top_3 = evaluate_baseline(test_sequences, markov_model, n=2, top_k=3)

print("\n Evaluation Results: ")
print(f"Baseline Top-1 Accuracy: {top_1:.4f}")
print(f"Baseline Top-3 Accuracy: {top_3:.4f}")